In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon
import numpy as np

In [ ]:
existing_cap_file = snakemake.input.existing_cap
onshore_shapefile = snakemake.input.onshore_shapefile
offshore_shapefile = snakemake.input.offshore_shapefile

year = snakemake.params.model_year
geodata_file = snakemake.output.geodata_file


In [ ]:
# get shapefiles from intermediate data from highres WF

geodata_files = {
    "onshore": onshore_shapefile,
    "offshore_bottom": offshore_shapefile,
}

# Load onshore shapefile
euroshape = gpd.read_file(geodata_files["onshore"]).set_index(["index"])
aggr_countries = euroshape.CNTR.unique().tolist()  # get relevant countries
aggr_countries

In [ ]:
# Calculate boundaries based on onshore and offshore shapefiles
boundaries = []
for geodata_file_name, geodata_file_path in geodata_files.items():
    boundaries.append(gpd.read_file(geodata_file_path).set_index(["index"]))

boundaries = pd.concat(boundaries).bounds

boundaries = boundaries.groupby(lambda x: "bountry").agg(
    {"minx": "min", "miny": "min", "maxx": "max", "maxy": "max"}
)

boundaries

In [ ]:
exist_pcap_dataset = pd.read_csv(existing_cap_file)

# convert to geopandas
existing_pcap = (
    gpd.GeoDataFrame(
        exist_pcap_dataset,
        geometry=gpd.points_from_xy(exist_pcap_dataset.lon, exist_pcap_dataset.lat),
        crs="epsg:4326",
    )
    .loc[
        :,
        [
            "Technology",
            "status",
            "iso_code",
            "capacity_MW",
            "geometry",
            "commissioning_year",
            "estimated_retirement_year",
        ],
    ]
    .query("iso_code==@aggr_countries")  # select only relevant countries
)
existing_pcap

In [ ]:
# Clip/Remove plants outside the considered area

rectx1 = boundaries.loc["bountry", "minx"]
rectx2 = boundaries.loc["bountry", "maxx"]
recty1 = boundaries.loc["bountry", "miny"]
recty2 = boundaries.loc["bountry", "maxy"]


polygon = Polygon(
    [
        (rectx1, recty1),
        (rectx1, recty2),
        (rectx2, recty2),
        (rectx2, recty1),
        (rectx1, recty1),
    ]
)
existing_pcap = gpd.clip(existing_pcap, polygon)

In [ ]:
#euroshape = euroshape.dissolve(by='CNTR_agg')

In [ ]:
# Determine the nuts2 region for every power plant
all_techs = gpd.sjoin(euroshape, existing_pcap, how="right", predicate="contains").loc[
    :,
    [
        "Technology",
        "status",
        "index",
        "CNTR",
        "iso_code",
        "capacity_MW",
        "geometry",
        "commissioning_year",
        "estimated_retirement_year",
    ],
]

# Matched plants
no_error_techs = (
    all_techs.query("CNTR == iso_code")  # country codes match
    # conserve "geometry" for plotting (it could be removed)
    .loc[
        :,
        [
            "Technology",
            "status",
            "iso_code",
            "index",
            "capacity_MW",
            "commissioning_year",
            "estimated_retirement_year",
            "geometry",
        ],
    ]
    # .loc[:, ["Technology", "iso_code", "index", "capacity_MW","geometry"]]
)

# Mismatched plants (country codes mismatch)
error_techs = all_techs.query("CNTR != iso_code")


# Plants with different country assignment or nan assignment
# from shapefile and database
# There are two groups:
# 1) Plants on the coast and offshore
# 2) Plants on the border of two countries


## Plants on the coast and offshore
# Solution: assign closest region"europe_offshore.geojson",
offshore_plants = error_techs.query("CNTR.isna()").to_crs(3035)  # nan values

for idx, row in offshore_plants.iterrows():
    # Find the closest region
    ztemp = euroshape.loc[euroshape["CNTR"] == row["iso_code"], :]
    dists = ztemp.to_crs(3035).distance(row.geometry)
    if len(dists) == 0:
        break
    offshore_plants.loc[idx, "index"] = dists.idxmin()

offshore_plants = (
    offshore_plants.to_crs(4326)
    # conserve "geometry" for plotting
    .loc[
        :,
        [
            "Technology",
            "status",
            "iso_code",
            "index",
            "capacity_MW",
            "commissioning_year",
            "estimated_retirement_year",
            "geometry",
        ],
    ]
)

## Plants on the border of two countries
# Solution: Assign information from shape file
# Issues spotted in Hungary/Austria/Czech Republic
# Only 3 solar plants with 1m resolutiongem_nuclear
country_border_plants = (
    error_techs.query("~CNTR.isna()")  # different country assignment
)
country_border_plants = (
    country_border_plants
    # conserve "geometry" for plotting
    .loc[
        :,
        [
            "Technology",
            "status",
            "CNTR",
            "index",
            "capacity_MW",
            "commissioning_year",
            "estimated_retirement_year",
            "geometry",
        ],
    ].rename(columns={"CNTR": "iso_code"})
)

### Merge all the plants
merged_techs = pd.concat(
    [
        no_error_techs,
        offshore_plants,
        country_border_plants,
    ]
).reset_index(drop=True)


In [ ]:
plants_for_model = merged_techs.loc[
    (merged_techs["status"] == "operating")
    & (merged_techs["estimated_retirement_year"] >= year)
].copy()

plants_gdf = gpd.GeoDataFrame(
    plants_for_model,
    geometry="geometry",
    crs="EPSG:4326",
)

plants_gdf.to_file(geodata_file, driver="GeoJSON")